# Week 2 — Data Preprocessing & Feature Engineering
Employee Attrition Prediction Using Machine Learning

**Student:** Rachit

This notebook implements the Week 2 preprocessing workflow. Model training is reserved for Week 3.

In [ ]:
# If needed, install dependencies in your environment:
# pip install pandas numpy matplotlib seaborn scikit-learn

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
RAW_PATH = "https://raw.githubusercontent.com/rachitgupt04/employee-attrition-prediction/main/WA_Fn-UseC_-HR-Employee-Attrition.csv"
PROCESSED_PATH = "data/processed/employee_attrition_preprocessed.csv"
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("results", exist_ok=True)


## 1. Load the dataset

Download the public IBM HR Analytics Employee Attrition & Performance CSV and place it at:

`data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv`

The commonly distributed dataset contains 1,470 rows and 35 columns.

In [ ]:
if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {RAW_PATH}. Download the public IBM HR Analytics Employee Attrition & Performance CSV "
        "and place it in data/raw/."
    )

df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nDescriptive statistics:")
display(df.describe(include="all").T)


## 2. Data quality checks

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print("Total missing values:", int(missing.sum()))
display(missing[missing > 0].to_frame("missing_count"))

print("Duplicate rows:", int(duplicates))

print("\nTarget distribution:")
display(df["Attrition"].value_counts())
display((df["Attrition"].value_counts(normalize=True)*100).round(2).rename("percentage"))


In [ ]:
# Unique values for categorical columns
categorical_cols = df.select_dtypes(include="object").columns
unique_summary = pd.DataFrame({
    "column": categorical_cols,
    "n_unique": [df[c].nunique(dropna=False) for c in categorical_cols]
}).sort_values("n_unique")
display(unique_summary)


## 3. Exploratory visualizations

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Attrition")
plt.title("Employee Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Employee Count")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(data=df, x="Age", bins=20, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome")
plt.title("Monthly Income by Attrition")
plt.xlabel("Attrition")
plt.ylabel("Monthly Income")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="OverTime", hue="Attrition")
plt.title("Attrition by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Employee Count")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
role_order = df["JobRole"].value_counts().index
sns.countplot(data=df, y="JobRole", hue="Attrition", order=role_order)
plt.title("Attrition by Job Role")
plt.xlabel("Employee Count")
plt.ylabel("Job Role")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="JobSatisfaction", hue="Attrition")
plt.title("Attrition by Job Satisfaction")
plt.xlabel("Job Satisfaction")
plt.ylabel("Employee Count")
plt.tight_layout()
plt.show()


In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number)
plt.figure(figsize=(14,10))
sns.heatmap(numeric_for_corr.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Numerical Features")
plt.tight_layout()
plt.show()


## 4. Cleaning and feature preparation

In [ ]:
data = df.copy()

# Drop identifier and constant columns if present.
drop_candidates = ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"]
drop_cols = [c for c in drop_candidates if c in data.columns]
data = data.drop(columns=drop_cols)

# Binary target encoding.
data["Attrition"] = data["Attrition"].map({"No": 0, "Yes": 1}).astype("int64")

print("Dropped columns:", drop_cols)
print("New shape:", data.shape)
display(data.head())


In [ ]:
# Define X and y
X = data.drop(columns=["Attrition"])
y = data["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("\nTrain target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTest target distribution:")
display(y_test.value_counts(normalize=True).rename("proportion"))


## 5. Leakage-resistant preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Fit only on training data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape :", X_test_processed.shape)


In [ ]:
# Inspect transformed feature names.
feature_names = preprocessor.get_feature_names_out()
print("Number of processed features:", len(feature_names))
print("First 30 processed features:")
print(feature_names[:30])


In [ ]:
# Save a fully numeric processed dataset for inspection.
# This file is useful for later work; Week 3 should still use the fitted pipeline to avoid leakage.
all_processed = preprocessor.transform(X)
processed_df = pd.DataFrame(
    all_processed.toarray() if hasattr(all_processed, "toarray") else all_processed,
    columns=feature_names
)
processed_df["Attrition"] = y.to_numpy()
processed_df.to_csv(PROCESSED_PATH, index=False)

print("Saved:", PROCESSED_PATH)
print("Processed dataset shape:", processed_df.shape)
display(processed_df.head())


## 6. Before vs After summary

**Before:** mixed numeric/categorical features, text target, identifiers/constants present.

**After:** target encoded as 0/1, non-informative identifiers/constants excluded, missing-value handling and scaling/encoding encapsulated in a reusable `ColumnTransformer` pipeline, and stratified train/test data prepared.

### Week 3 handoff
Use `preprocessor` inside a model `Pipeline` when training Logistic Regression, Decision Tree and Random Forest. Keep the test set untouched until final evaluation.